In [59]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
# from sklearn.datasets import make_blobs, make_moons
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest

df = pd.read_csv('Pareto_data.csv')
df

,Formula,lat,M_s,T_C,Pugh,Cauchy,norm_M_s,norm_T_C,norm_Pugh,norm_Cauchy,HV
0,Sc0.247Cr0.153Fe0.270Co0.268Zn0.062,0,0.279427,781.209090,2.009888,18.667694,0.275120,0.746610,-1.246701,-0.395692,41.729279
1,V0.092Fe0.250Co0.235Cu0.289Zn0.134,0,0.316815,1074.082518,2.655984,29.667529,0.455034,1.371003,0.455019,0.321549,112.331764
2,Sc0.236Cr0.171Fe0.334Co0.196Zn0.063,0,0.281833,776.263777,1.974558,17.197633,0.286698,0.736067,-1.339756,-0.491547,39.809824
3,Sc0.320Ti0.162V0.126Fe0.336Zn0.056,0,0.119123,746.171017,1.862694,5.640815,-0.496259,0.671910,-1.634390,-1.245105,13.685475
4,Sc0.207Cr0.109Fe0.382Co0.203Zn0.099,0,0.348728,1061.110078,1.948310,13.425157,0.608596,1.343346,-1.408889,-0.737530,60.925355
...,...,...,...,...,...,...,...,...,...,...,...
545,Mn0.073Fe0.655Co0.117Ni0.069Cu0.086,0,0.761947,1341.220030,1.967914,13.662745,2.597000,1.940528,-1.357254,-0.722038,165.095158
546,Ti0.047Fe0.465Co0.292Cu0.177Zn0.019,0,0.609847,1730.713745,2.321051,31.968527,1.865099,2.770911,-0.427143,0.471585,277.424624
547,Ti0.061Fe0.483Co0.302Cu0.061Zn0.093,0,0.594674,1701.107667,2.087840,20.548423,1.792087,2.707792,-1.041388,-0.273059,197.182384
548,Mn0.022Fe0.195Co0.447Ni0.191Cu0.145,1,0.531013,966.063086,2.377813,21.753443,1.485752,1.140710,-0.277641,-0.194487,128.400043


In [60]:
np.random.RandomState(0)
random_state = 1
outliers_fraction = 0.02
# 处理离群值
anomaly_algorithms = [
     EllipticEnvelope(contamination=outliers_fraction, 
                      random_state=random_state),
     # OneClassSVM(nu=outliers_fraction, kernel="rbf", 
     #             gamma=10),
     IsolationForest(contamination=outliers_fraction, 
                     random_state=random_state)]
X = df[['lat', 'M_s', 'T_C', 'Pugh', 'Cauchy']]


In [61]:
results = {}
for model in anomaly_algorithms:
    model_name = model.__class__.__name__
    model.fit(X)
    predictions = model.predict(X)
    # In anomaly detection, -1 typically indicates an outlier
    results[model_name] = np.where(predictions == -1)[0]
    print(df.iloc[results[model_name]])
    df.iloc[results[model_name]].to_csv(f'{model_name}_outliers.csv')
# results

                                 Formula  lat       M_s         T_C      Pugh  \
14    Sc0.138V0.126Mn0.413Co0.197Zn0.126    0  0.296819  188.764460  0.895650   
158   Sc0.191Ti0.172V0.171Cr0.233Zn0.233    1  0.000000    0.000000  5.799876   
245  Mn0.334Co0.250Ni0.119Cu0.117Zn0.180    0  0.504217  198.161192  2.108655   
252  Mn0.336Fe0.208Co0.099Cu0.068Zn0.289    0  0.520747   42.182290  1.801148   
262  Sc0.171Mn0.337Fe0.299Ni0.152Zn0.041    0  0.529014  217.510731  1.653333   
283  Mn0.250Co0.297Ni0.229Cu0.144Zn0.080    1  0.457202   71.895569  2.496386   
309  Sc0.111Mn0.348Fe0.108Cu0.116Zn0.317    1  0.438828   40.810835  2.275621   
365  Mn0.293Fe0.209Co0.263Ni0.178Zn0.057    0  0.637734  570.997210  1.765443   
375  Mn0.271Fe0.224Co0.228Ni0.208Zn0.069    0  0.629948  536.721607  1.830997   
441   Sc0.091Ti0.091V0.203Cr0.120Zn0.495    0  0.000000    0.000000  1.435165   
506   Sc0.264Ti0.085V0.181Cr0.264Zn0.206    1  0.000000    0.000000  5.128974   

        Cauchy  norm_M_s  n

In [62]:
all_outlier_indices = set(
    results['EllipticEnvelope']).union(
    results['IsolationForest'])

# Remove these indices from the original dataset
data_cleaned = df.drop(index=all_outlier_indices)
data_cleaned.describe()

,lat,M_s,T_C,Pugh,Cauchy,norm_M_s,norm_T_C,norm_Pugh,norm_Cauchy,HV
count,532.000000,532.000000,532.000000,532.000000,532.000000,532.000000,532.000000,532.000000,532.000000,532.000000
mean,0.263158,0.219364,437.499439,2.486073,25.727728,-0.013898,0.013836,0.007500,0.064656,47.983210
std,0.440762,0.206120,473.740528,0.305163,13.083531,0.991844,1.009994,0.803756,0.853108,69.338492
min,0.000000,0.000000,0.000000,1.582239,-8.104911,-1.069476,-0.918893,-2.373066,-2.141391,0.083451
25%,0.000000,0.012943,2.942207,2.286468,17.246166,-1.007193,-0.912620,-0.518232,-0.488382,0.452472
50%,0.000000,0.190145,286.945318,2.508879,25.835952,-0.154501,-0.307138,0.067567,0.071712,17.492921
75%,1.000000,0.344024,718.298783,2.697268,33.862658,0.585960,0.612488,0.563757,0.595091,68.737828
max,1.000000,0.792449,1763.406171,3.273654,70.216765,2.743778,2.840610,2.081872,2.965549,366.804828
